In [ ]:
!pip install ortools

In [ ]:
from ortools.sat.python import cp_model

#First example

In [ ]:
#crear el modelo CSP
model = cp_model.CpModel()

In [ ]:
#variables y dominios
x = model.NewIntVar(0,2,'x')
y = model.NewIntVar(0,2,'y')
z = model.NewIntVar(0,2,'z')
#restricciones
model.Add(x != y)
model.Add(y == z + 1)

In [ ]:
#crear el solver
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL:
  print('x = {:d}'.format(solver.Value(x)))
  print('y = {:d}'.format(solver.Value(y)))
  print('z = {:d}'.format(solver.Value(z)))

x = 0
y = 1
z = 0


#Cryptogram

##Brute-force

In [ ]:
def cryptogramBF():
  for s in range(1,10):
    for e in range(10):
      for n in range(10):
        for d in range(10):
          for m in range(1,10):
            for o in range(10):
              for r in range(10):
                for y in range(10):
                  l = [s,e,n,d,m,o,r,y]
                  c = set(l)
                  if len(l) == len(c) and 1000*(s + m) + 100*(e + o) + 10*(n + r) + (d + e) == 10000*m + 1000*o + 100*n + 10*e + y:
                    print(' ',s,e,n,d,'+')
                    print(' ',m,o,r,e)
                    print('---------')
                    print(m,o,n,e,y)
cryptogramBF()

  9 5 6 7 +
  1 0 8 5
---------
1 0 6 5 2


In [ ]:
#crear CSP
model = cp_model.CpModel()
#variables y dominios
s = model.NewIntVar(1,9,'s')
e = model.NewIntVar(0,9,'e')
n = model.NewIntVar(0,9,'n')
d = model.NewIntVar(0,9,'d')
m = model.NewIntVar(1,1,'m')
o = model.NewIntVar(0,9,'o')
r = model.NewIntVar(0,9,'r')
y = model.NewIntVar(0,9,'y')
v = [s,e,n,d,m,o,r,y]
#restricciones
model.Add(1000 * (s+m) + 100 * (e+o) + 10 * (n+r) + (d+e) == 10000*m+1000*o+100*n+10*e+y)
model.AddAllDifferent(v)

In [ ]:
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL:
  s,e,n,d,m,o,r,y = solver.Value(s),solver.Value(e),solver.Value(n),solver.Value(d),solver.Value(m),solver.Value(o),solver.Value(r),solver.Value(y)
  print(' ',s,e,n,d,'+')
  print(' ',m,o,r,e)
  print('---------')
  print(m,o,n,e,y)

  9 5 6 7 +
  1 0 8 5
---------
1 0 6 5 2


In [ ]:
#print(solver.StatusName(4))

##Latin Square

In [ ]:
n = 9
#crear CSP
model = cp_model.CpModel()
#variables y dominios
grilla = []
for i in range(n):
  fila = []
  for j in range(n):
    fila += [model.NewIntVar(1,n,'x'+str(i)+str(j))]
  #fila = [model.NewIntVar(1,n,'x'+str(i)+str(j)) for j in range(n)]
  grilla += [fila]
#print(grilla)
#restricciones
##toda fila tiene valores distintos
for i in range(n):
  model.AddAllDifferent(grilla[i])
##toda columna tiene valores distintos
for j in range(n):
  #recuperar columna
  c = [grilla[i][j] for i in range(n)]
  #print(c)
  model.AddAllDifferent(c)

In [ ]:
solver = cp_model.CpSolver()
status = solver.Solve(model)
if status == cp_model.OPTIMAL:
  for i in range(n):
    for j in range(n):
      print(solver.Value(grilla[i][j]),end=' ')
    print()

1 9 4 2 7 6 5 8 3 
9 4 8 3 5 7 6 2 1 
7 1 3 9 4 2 8 5 6 
6 7 1 5 3 9 2 4 8 
5 8 6 1 9 4 3 7 2 
4 5 2 8 6 1 9 3 7 
3 2 9 4 1 8 7 6 5 
2 3 7 6 8 5 1 9 4 
8 6 5 7 2 3 4 1 9 


#Magic Square

In [ ]:
n = 4
s = int((n**2)*(1+n**2)/(2*n))
print(s)
#crear CSP
model = cp_model.CpModel()
#variables y dominios
grilla = []
for i in range(n):
  fila = []
  for j in range(n):
    fila += [model.NewIntVar(1,n**2,'x'+str(i)+str(j))]
  grilla += [fila]
#print(grilla)
#restricciones
##recuperar todos los valores de la grilla en un solo arreglo
v = []
for i in range(n):
  v += grilla[i]
#print(v)
model.AddAllDifferent(v)
##todas las filas deben sumar el mismo valor
for i in range(n):
  model.Add(sum(grilla[i]) == s)
##todas las columnas deben sumar el mismo valor
for j in range(n):
  c = [grilla[i][j] for i in range(n)]
  model.Add(sum(c) == s)

34


In [ ]:
#one solution
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL:
  for i in range(n):
    for j in range(n):
      print(solver.Value(grilla[i][j]),end=' ')
    print()

1 16 10 7 
15 2 8 9 
14 3 11 6 
4 13 5 12 


In [ ]:
class VarArraySolutionPrinter(cp_model.CpSolverSolutionCallback):
    """Print intermediate solutions."""

    def __init__(self, variables):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.__variables = variables
        self.__solution_count = 0

    def on_solution_callback(self):
        self.__solution_count += 1
        for v in self.__variables:
            print('%s=%i' % (v, self.Value(v)), end=' ')
        print()

    def solution_count(self):
        return self.__solution_count

In [ ]:
#all solutions
solver = cp_model.CpSolver()
solution_printer = VarArraySolutionPrinter(v)
status = solver.SearchForAllSolutions(model,solution_printer)
print(solution_printer.solution_count())

#Winter school

In [ ]:
#crear CSP
model = cp_model.CpModel()
#variables y dominios
sesion = []
for i in range(11): #sesiones
  curso = []
  for d in range(2): #dias
    dias =[]
    for t in range(2): #turnos
      turno = []
      for s in range(3): #salones
        turno += [model.NewBoolVar('x'+str(i)+'_'+str(d)+'_'+str(t)+'_'+str(s))]
      dias += [turno]
    curso += [dias]
  sesion += [curso]
print(sesion)
#restricciones
##cada curso se dicta 1 vez
for i in range(11):
  l = []
  for d in range(2):
    for t in range(2):
      for s in range(3):
        l += [sesion[i][d][t][s]]
  #print(l)
  model.Add(sum(l) == 1)
##cada salon solo hasta 1 curso al mismo tiempo
for d in range(2):
  for t in range(2):
    for s in range(3):
      l = [sesion[i][d][t][s] for i in range(11)]
      model.Add(sum(l) <= 1)
##no se pueden hacer en simultaneo ciertas sesiones
def no_simultaneo(m, a, b):
  for d in range(2):
    for t in range(2):
      m.Add(sum(sesion[a][d][t]) + sum(sesion[b][d][t]) <= 1)

#A B C D E F G H I J K
#0 1 2 3 4 5 6 7 8 9 10

#A y J
no_simultaneo(model, 0, 9)
#J e I
no_simultaneo(model, 9, 8)
#C y F
no_simultaneo(model, 2, 5)
#F and G
no_simultaneo(model, 5, 6)
# D and H
no_simultaneo(model, 3, 7)
# B and D
no_simultaneo(model, 1, 3)
# K and E
no_simultaneo(model, 10, 4)
#B, I, H and G
no_simultaneo(model, 1, 8) #B,I
no_simultaneo(model, 1, 7) #B,H
no_simultaneo(model, 1, 6) #B,G
no_simultaneo(model, 8, 7) #I,H
no_simultaneo(model, 8, 6) #I,G
no_simultaneo(model, 7, 6) #H,G
#A, G and E
no_simultaneo(model, 0, 6) #A,G
no_simultaneo(model, 0, 4) #A,E
no_simultaneo(model, 6, 4) #G,E
#B, H and K
no_simultaneo(model, 1, 7) #B,H
no_simultaneo(model, 1, 10) #B,K
no_simultaneo(model, 7, 10) #H,K
#A, B, C and H
no_simultaneo(model, 0, 1) #A,B
no_simultaneo(model, 0, 2) #A,C
no_simultaneo(model, 0, 7) #A,H
no_simultaneo(model, 1, 2)  #B,C
no_simultaneo(model, 1, 7) #B,H
no_simultaneo(model, 2, 7) #C,H
#D, F and J
no_simultaneo(model, 3, 5) #D,F
no_simultaneo(model, 3, 9) #D,J
no_simultaneo(model, 5, 9) #F,J

def before(m,a,b):
  #evento a no puede ser el 1er día en el 1er turno
  m.Add(sum(sesion[b][0][0]) == 0)
  #si el evento a no es el 1er día en la mañana
  #entonces el evento b no puede ser el 1er día en la tarde
  m.Add(sum(sesion[a][0][0]) >= sum(sesion[b][0][1]))
  #si el evento a no se da el 1er día en la mañana o en la tarde
  #entonces el evento b no puede ser el 2do día en la tarde
  m.Add(sum(sesion[a][0][0]) + sum(sesion[a][0][1]) >= sum(sesion[b][1][0]))
  #si el evento a no es el 2do día en la mañana
  #entonces el evento b no puede ser el 2do día en la tarde
  m.Add(sum(sesion[a][1][0]) >= sum(sesion[b][1][1]))

#E before J
before(model, 4, 9)
#model.Add(sum(sesion[9][0][0]) == 0)
#model.Add(sum(sesion[4][0][0]) >= sum(sesion[9][0][1]))
#model.Add(sum(sesion[4][0][0]) + sum(sesion[4][0][1]) >= sum(sesion[9][1][0]))
#model.Add(sum(sesion[4][1][0]) >= sum(sesion[9][1][1]))

#D and F before K.
before(model, 3, 10)
#before(model, 5, 10)

[[[[x0_0_0_0(0..1), x0_0_0_1(0..1), x0_0_0_2(0..1)], [x0_0_1_0(0..1), x0_0_1_1(0..1), x0_0_1_2(0..1)]], [[x0_1_0_0(0..1), x0_1_0_1(0..1), x0_1_0_2(0..1)], [x0_1_1_0(0..1), x0_1_1_1(0..1), x0_1_1_2(0..1)]]], [[[x1_0_0_0(0..1), x1_0_0_1(0..1), x1_0_0_2(0..1)], [x1_0_1_0(0..1), x1_0_1_1(0..1), x1_0_1_2(0..1)]], [[x1_1_0_0(0..1), x1_1_0_1(0..1), x1_1_0_2(0..1)], [x1_1_1_0(0..1), x1_1_1_1(0..1), x1_1_1_2(0..1)]]], [[[x2_0_0_0(0..1), x2_0_0_1(0..1), x2_0_0_2(0..1)], [x2_0_1_0(0..1), x2_0_1_1(0..1), x2_0_1_2(0..1)]], [[x2_1_0_0(0..1), x2_1_0_1(0..1), x2_1_0_2(0..1)], [x2_1_1_0(0..1), x2_1_1_1(0..1), x2_1_1_2(0..1)]]], [[[x3_0_0_0(0..1), x3_0_0_1(0..1), x3_0_0_2(0..1)], [x3_0_1_0(0..1), x3_0_1_1(0..1), x3_0_1_2(0..1)]], [[x3_1_0_0(0..1), x3_1_0_1(0..1), x3_1_0_2(0..1)], [x3_1_1_0(0..1), x3_1_1_1(0..1), x3_1_1_2(0..1)]]], [[[x4_0_0_0(0..1), x4_0_0_1(0..1), x4_0_0_2(0..1)], [x4_0_1_0(0..1), x4_0_1_1(0..1), x4_0_1_2(0..1)]], [[x4_1_0_0(0..1), x4_1_0_1(0..1), x4_1_0_2(0..1)], [x4_1_1_0(0..1), x4_1

In [ ]:
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL:
  for i in range(11):
    print(i,end='\t')
    for d in range(2):
      for t in range(2):
        for s in range(3):
          print(solver.Value(sesion[i][d][t][s]),end=' ')
        print("",end='- ')
      print("",end="\t| ")
    print()

  print("-----------")
  for d in range(2):
    print('d'+str(d+1),'\t',end=' ')
    for t in range(2):
      for s in range(2):
        for i in range(11):
          if solver.Value(sesion[i][d][t][s]) == 1:
            print("curso"+str(i), end=' ')
      print('',end='|')
    print()

0	1 0 0 - 0 0 0 - 	| 0 0 0 - 0 0 0 - 	| 
1	0 0 0 - 0 0 0 - 	| 0 0 0 - 1 0 0 - 	| 
2	0 0 0 - 0 1 0 - 	| 0 0 0 - 0 0 0 - 	| 
3	0 0 1 - 0 0 0 - 	| 0 0 0 - 0 0 0 - 	| 
4	0 0 0 - 0 0 0 - 	| 0 0 1 - 0 0 0 - 	| 
5	0 0 0 - 0 0 0 - 	| 1 0 0 - 0 0 0 - 	| 
6	0 0 0 - 0 0 1 - 	| 0 0 0 - 0 0 0 - 	| 
7	0 0 0 - 0 0 0 - 	| 0 1 0 - 0 0 0 - 	| 
8	0 1 0 - 0 0 0 - 	| 0 0 0 - 0 0 0 - 	| 
9	0 0 0 - 0 0 0 - 	| 0 0 0 - 0 1 0 - 	| 
10	0 0 0 - 1 0 0 - 	| 0 0 0 - 0 0 0 - 	| 
-----------
d1 	 curso0 curso8 |curso10 curso2 |
d2 	 curso5 curso7 |curso1 curso9 |


#Anagrama

In [ ]:
from ortools.sat.python import cp_model

class VarArraySolutionPrinter(cp_model.CpSolverSolutionCallback):
    def __init__(self, variables):
        super().__init__()
        self.__variables = variables
        self.__solution_count = 0

    def on_solution_callback(self):
        self.__solution_count += 1
        for v in self.__variables:
          #print(f"{v}={self.Value(v)}", end=" ")
          if self.Value(v) == 1:
            print(f"{v}={self.Value(v)}", end=" ")
        print()

    @property
    def solution_count(self):
        return self.__solution_count

def resolver_anagrama(letras_objetivo, diccionario):
    # Convertir las letras objetivo a minúsculas y contar sus frecuencias
    letras_objetivo = letras_objetivo.lower()
    conteo_objetivo = {c: letras_objetivo.count(c) for c in set(letras_objetivo)}
    longitud_objetivo = len(letras_objetivo)

    # Filtrar el diccionario: solo nos interesan palabras con la misma longitud
    palabras_candidatas = [p.lower() for p in diccionario if len(p) == longitud_objetivo]
    if not palabras_candidatas:
        return []

    # Inicializar el modelo CP-SAT
    model = cp_model.CpModel()

    # Variables de decisión: una variable booleana por cada palabra candidata
    # El solver decidirá cuál de estas variables se activa (1) o desactiva (0)
    print(palabras_candidatas)
    palabras_vars = [model.NewBoolVar(f'w_{i}') for i in range(len(palabras_candidatas))]

    # Restricción 1: Las frecuencias de cada letra de la palabra elegida deben coincidir con el objetivo
    # Para cada letra del alfabeto presente en nuestra palabra objetivo
    for letra, max_conteo in conteo_objetivo.items():
      # Calculamos cuántas veces aparece esta letra en cada palabra candidata
      conteos_en_candidatas = [palabra.count(letra) for palabra in palabras_candidatas]
      print(letra, max_conteo, conteos_en_candidatas)
      # El producto escalar asegura que la palabra elegida sume exactamente los caracteres requeridos
      model.Add(sum(palabras_vars[i] * conteos_en_candidatas[i] for i in range(len(palabras_candidatas))) == max_conteo)

    # Invocar al solver de OR-Tools
    solver = cp_model.CpSolver()
    solution_printer = VarArraySolutionPrinter(palabras_vars)
    status = solver.SearchForAllSolutions(model,solution_printer)
    #status = solver.Solve(model)

    # Buscar y retornar todas las soluciones válidas
    soluciones = []
    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        # Si quieres encontrar múltiples soluciones (en caso de homógrafos/más anagramas),
        # podemos iterar filtrando las ya encontradas o usar un SolutionCallback.
        # Aquí extraemos de forma simple las palabras que cumplen la condición:
        #print([solver.Value(var) for var in palabras_vars])
        for i, var in enumerate(palabras_vars):
            if solver.Value(var) == 1:
                soluciones.append(palabras_candidatas[i])
    #return soluciones

# --- Prueba del algoritmo ---
# Un pequeño diccionario de ejemplo
diccionario_espanol = ["roma", "amor", "ramo", "mora", "perro", "casa", "omar", "riesgo"]
letras_desordenadas = "sergio"

resolver_anagrama(letras_desordenadas, diccionario_espanol)

['riesgo']
r 1 [1]
o 1 [1]
e 1 [1]
i 1 [1]
g 1 [1]
s 1 [1]
w_0=1 
